In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import RelaxedOneHotCategorical
import time
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr, entropy, ttest_ind
from sklearn.metrics import mean_absolute_error, r2_score, confusion_matrix

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (15, 10)

In [ ]:
def simulate_one_election_np(num_districts=100, voters_per_district=100000,
                             alpha=None, beta=None, alpha_params=[1,1,1], seed=None):
    rng = np.random.default_rng(seed)
    if alpha is None:
        alpha = rng.dirichlet(alpha_params)
    if beta is None:
        beta = rng.uniform(0, 1)

    district_winners = []
    votes_per_district = []

    for _ in range(num_districts):
        votes = np.zeros(3, dtype=int)
        for _ in range(voters_per_district):
            if votes.sum() == 0:
                probs = alpha
            else:
                share = votes / votes.sum()
                probs = beta * share + (1 - beta) * alpha
            choice = rng.choice(3, p=probs)
            votes[choice] += 1
        winner = np.argmax(votes)
        district_winners.append(winner)
        votes_per_district.append(votes)

    counts = np.bincount(district_winners, minlength=3)
    votes_per_district = np.array(votes_per_district)
    return counts, alpha, beta, votes_per_district

In [2]:
def differentiable_election_torch_batch(alpha_batch, beta_batch, num_districts, voters_per_district,
                                        device='cpu', gumbel_temp=0.5):
    """
    Improved differentiable election simulator with better gradient flow
    """
    B = alpha_batch.shape[0]
    alpha_exp = alpha_batch.unsqueeze(1).expand(-1, num_districts, -1).contiguous()
    beta_exp = beta_batch.unsqueeze(1).expand(-1, num_districts).contiguous()
    soft_counts = torch.zeros((B, num_districts, 3), device=device, dtype=torch.float32)

    for v in range(voters_per_district):
        sums = soft_counts.sum(dim=2, keepdim=True)
        # Use a small epsilon to avoid division by zero
        share = soft_counts / (sums + 1e-6)

        # Compute probabilities for this voter
        # When sum is zero, use alpha; otherwise use mixture
        probs = beta_exp.unsqueeze(2) * share + (1.0 - beta_exp).unsqueeze(2) * alpha_exp

        # Handle first voter case (when sum is 0)
        mask_zero = (sums.squeeze(2) < 0.1).unsqueeze(2)
        probs = torch.where(mask_zero, alpha_exp, probs)

        # Ensure valid probability distribution: non-negative and sums to 1
        probs = torch.relu(probs)  # Ensure non-negativity
        # Add a small epsilon to the probabilities before normalization
        # to prevent issues with zero probabilities and ensure numerical stability.
        probs = probs + 1e-10
        # Normalize to ensure the sum of probabilities for each row is exactly 1
        probs = probs / probs.sum(dim=2, keepdim=True)
        # A final clamp to ensure values are within [0, 1] range after normalization,
        # mainly to catch any extreme floating point errors.
        # probs = probs.clamp(min=0.0, max=1.0) # Removed this line as it can cause sum-to-one violations

        # Sample using Gumbel-Softmax
        probs_flat = probs.view(-1, 3)
        dist = RelaxedOneHotCategorical(temperature=gumbel_temp, probs=probs_flat)
        sample_flat = dist.rsample()
        sample = sample_flat.view(B, num_districts, 3)
        soft_counts = soft_counts + sample

    return soft_counts

In [3]:
import torch.nn as nn

class EncoderNN(nn.Module):
    def __init__(self, num_districts, hidden_dims=(256, 128, 64)):
        super().__init__()
        self.input_dim = num_districts * 3

        # Deeper network with batch normalization
        layers = []
        prev = self.input_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.BatchNorm1d(h))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(0.1))
            prev = h

        self.mlp = nn.Sequential(*layers)

        # Separate heads for alpha and beta
        self.alpha_head = nn.Sequential(
            nn.Linear(prev, 32),
            nn.ReLU(),
            nn.Linear(32, 3)
        )

        self.beta_head = nn.Sequential(
            nn.Linear(prev, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, phi, voters_per_district):
        # Normalize input to [0, 1] range
        #x = phi.reshape(phi.shape[0], -1) / voters_per_district  # Use reshape instead of view
        x = phi.reshape(phi.shape[0], -1) / 1
        h = self.mlp(x)
        alpha_logits = self.alpha_head(h)
        alpha = torch.softmax(0.25*alpha_logits, dim=-1)
        beta = 0.5 + 0.5*torch.sigmoid(self.beta_head(h).squeeze(-1))
        return alpha, beta, alpha_logits

In [4]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

def district_share_variance(district_votes: torch.Tensor, S: int) -> torch.Tensor:
    eps     = 1e-8
    #max_var = 1.0 / S
    max_var = 1.0
    totals  = district_votes.sum(dim=2, keepdim=True).clamp(min=eps)
    share   = district_votes / totals
    mean    = share.mean(dim=2, keepdim=True)
    var     = ((share - mean) ** 2).mean(dim=2)
    return (var / max_var).clamp(0.0, 1.0)

def party_share_variance(district_votes: torch.Tensor, K: int) -> torch.Tensor:
    eps = 1e-8
    #max_var = 1.0 / K
    max_var = 1.0
    totals = district_votes.sum(dim=1, keepdim=True).clamp(min=eps)
    shares = district_votes / totals
    mean = shares.mean(dim=1, keepdim=True)
    var = ((shares - mean) ** 2).mean(dim=1)
    return (var / max_var).clamp(0.0, 1.0)

def party_seat_share(district_votes: torch.Tensor) -> torch.Tensor:
    num_examples = district_votes.shape[0]
    num_districts = district_votes.shape[1]
    num_parties = district_votes.shape[2]
    theta = [];
    for i in range(num_examples):
        seats = np.zeros(num_parties)
        for j in range(num_districts):
            # Move the tensor to CPU and convert to NumPy array before using np.argmax
            votes = district_votes[i, j, :].detach().cpu().numpy()
            k = np.argmax(votes)
            seats[k] += 1
        theta.append(seats/num_districts)
    return np.array(theta)

In [ ]:
ce_loss = nn.CrossEntropyLoss()
mse_loss = nn.MSELoss()
p1 = [0.5,0.3,0.2]
p2 = [0.4,0.4,0.2]
p3 = [0.8,0.1,0.1]

loss1 = mse_loss(torch.tensor(p2), torch.tensor(p1))
loss2 = mse_loss(torch.tensor(p3), torch.tensor(p1))
print(loss1)
print(loss2)

tensor(0.0067)
tensor(0.0467)


In [5]:
import os

#CKPT_DIR = "/content/drive/MyDrive/gradDPM_checkpoints"
#os.makedirs(CKPT_DIR, exist_ok=True)

def train_encoder_decoder(X, true_params, num_examples, num_districts, voters_per_district, num_parties,
                          device='cpu', epochs=50, batch_size=8, lr=1e-3,
                          gumbel_temp=0.5, verbose=True):
    torch.manual_seed(0)
    #X = torch.tensor(X_np, dtype=torch.float32, device=device)
    alpha_true = torch.tensor(np.array([p[0] for p in true_params]), dtype=torch.float32, device=device)
    beta_true = torch.tensor(np.array([p[1] for p in true_params]), dtype=torch.float32, device=device)
    #theta_true = torch.tensor(np.array([p[0] for p in true_params]), dtype=torch.float32, device=device)

    # num_district = num_districts.min()
    num_district = num_districts

    model = EncoderNN(num_districts=num_district).to(device)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

    mse_loss = nn.MSELoss()
    ce_loss = nn.CrossEntropyLoss()

    indices = np.arange(num_examples)
    start_time = time.time()

    loss_history = {'total': [], 'reconstruction': [], 'variance': [], 'alpha': [], 'beta': [], 'theta': [], 'param_direct': []}

    best_loss = float('inf')

    for ep in range(1, epochs + 1):
        model.train()
        np.random.shuffle(indices)
        epoch_loss_total = 0.0
        epoch_loss_recon = 0.0
        epoch_loss_alpha = 0.0
        epoch_loss_beta = 0.0
        epoch_loss_theta = 0.0
        epoch_loss_param = 0.0
        epoch_loss_variance = 0.0

        indices = np.arange(num_examples)
        for i in range(0, num_examples, batch_size):
            batch_idx = indices[i:i+batch_size]
            X_range = []
            for j in batch_idx:
                x = X[j]
                X_range.append(x)
            phi_original = torch.tensor(np.stack(X_range), dtype=torch.float32, device=device)

            #phi_original = X_range
            alpha_true_b = alpha_true[batch_idx]
            beta_true_b = beta_true[batch_idx]

            # Encoder predicts alpha, beta
            alpha_pred, beta_pred, alpha_logits = model(phi_original, voters_per_district)

            # Decoder reconstructs phi
            phi_reconstructed = differentiable_election_torch_batch(
                alpha_pred, beta_pred,
                num_districts=num_district,
                voters_per_district=voters_per_district,
                device=device,
                gumbel_temp=gumbel_temp
            )

            x_svar_true = district_share_variance(phi_original, num_district)
            x_kvar_true = party_share_variance(phi_original, num_parties)
            theta_true = party_seat_share(phi_original)

            x_svar_reconstructed = district_share_variance(phi_reconstructed, num_district)
            x_kvar_reconstructed = party_share_variance(phi_reconstructed, num_parties)
            theta_reconstructed = party_seat_share(phi_reconstructed)

            # MULTI-COMPONENT LOSS:
            # 1. Reconstruction loss (MSE on normalized votes)
            loss_recon = mse_loss(phi_reconstructed / voters_per_district, #compare full election results
                                 phi_original / voters_per_district)

            loss_variance = mse_loss(x_svar_reconstructed, x_svar_true) + mse_loss(x_kvar_reconstructed, x_kvar_true)
            #loss_recon += loss_variance

            # 2. Direct parameter supervision (helps with gradient flow)
            target_cls = torch.argmax(alpha_true_b, dim=1)
            #loss_alpha1_direct = ce_loss(alpha_logits, target_cls)  #predict winning party
            #loss_alpha2_direct = ce_loss(alpha_true_b, alpha_pred)
            loss_alpha3_direct = mse_loss(alpha_true_b, alpha_pred)
            loss_beta_direct = mse_loss(beta_pred, beta_true_b)    #predict ABM parameter
            loss_theta_direct = mse_loss(torch.tensor(theta_reconstructed), torch.tensor(theta_true))

            #loss_param_direct = loss_alpha1_direct + loss_alpha2_direct + loss_alpha3_direct + loss_beta_direct + loss_theta_direct
            loss_param_direct = loss_alpha3_direct + loss_beta_direct + loss_theta_direct

            # 3. Combined loss with weighting
            # Start with more direct supervision, gradually shift to reconstruction
            #param_weight = max(0.1, 1.0 - ep / epochs)  # Decay from 1.0 to 0.1
            #recon_weight = 1.0 - param_weight + 0.1

            param_weight = 4.0
            recon_weight = 1.0
            variance_weight = 3.0

            loss = recon_weight * loss_recon + param_weight * loss_param_direct + variance_weight * loss_variance

            # Backprop
            optimizer.zero_grad()
            loss.backward()

            # Gradient clipping to prevent explosions
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            optimizer.step()

            # Track losses
            epoch_loss_total += loss.item() * phi_original.shape[0]
            epoch_loss_recon += loss_recon.item() * phi_original.shape[0]
            epoch_loss_variance += loss_variance.item() * phi_original.shape[0]
            epoch_loss_param += loss_param_direct.item() * phi_original.shape[0]
       #     epoch_loss_alpha += loss_alpha1_direct.item() * phi_original.shape[0]
       #     epoch_loss_alpha += loss_alpha2_direct.item() * phi_original.shape[0]
            epoch_loss_alpha += loss_alpha3_direct.item() * phi_original.shape[0]
            epoch_loss_beta += loss_beta_direct.item() * phi_original.shape[0]
            epoch_loss_theta += loss_theta_direct.item() * phi_original.shape[0]

        # Average losses
        epoch_loss_total /= num_examples
        epoch_loss_recon /= num_examples
        epoch_loss_variance /= num_examples
        epoch_loss_param /= num_examples
        epoch_loss_alpha /= num_examples
        epoch_loss_beta /= num_examples
        epoch_loss_theta /= num_examples

        loss_history['total'].append(epoch_loss_total)
        loss_history['reconstruction'].append(epoch_loss_recon)
        loss_history['variance'].append(epoch_loss_variance)
        loss_history['param_direct'].append(epoch_loss_param)
        loss_history['alpha'].append(epoch_loss_alpha)
        loss_history['beta'].append(epoch_loss_beta)
        loss_history['theta'].append(epoch_loss_theta)

        # Update learning rate
        scheduler.step(epoch_loss_total)

        #if verbose and (ep % 10 == 0 or ep == 1):
        if verbose :
            print(f"Epoch {ep:04d} | Total: {epoch_loss_total:.4f} | "
                  f"Recon: {epoch_loss_recon:.4f} | Param: {epoch_loss_param:.4f} | "
                  f"Variance: {epoch_loss_variance:.4f} | "
                  f"Alpha: {epoch_loss_alpha:.4f} | Beta: {epoch_loss_beta:.4f} | "
                  f"Theta: {epoch_loss_theta:.4f} | "
                  f"LR: {optimizer.param_groups[0]['lr']:.6f} | "
                  f"elapsed {time.time()-start_time:.1f}s")

        # Save best model
        if epoch_loss_total < best_loss:
            best_loss = epoch_loss_total

  #          torch.save({
  #      'epoch': ep,
  #      'model_state_dict': model.state_dict(),
  #      'optimizer_state_dict': optimizer.state_dict(),
  #      'scheduler_state_dict': scheduler.state_dict(),
  #      'loss': best_loss,
  #      'loss_history': loss_history,
  #      'num_districts': num_districts,
  #      'voters_per_district': voters_per_district
  #  }, f"{CKPT_DIR}/best_model.pt")

    return model, loss_history

In [ ]:
import numpy as np

def make_synthetic_dataset(num_examples=500, seed=0):
    rng = np.random.default_rng(seed)
    X, Y, true_params, num_districts, voters_per_districts = [], [], [], [], []
    for _ in range(num_examples):
        alpha = rng.dirichlet([1, 1, 1])
        beta = rng.uniform(0.5, 1)
        num_district = rng.integers(50, 200)
        voters_per_district = rng.integers(5000, 15000)
        counts, _, _, votes_per_district = simulate_one_election_np(
            num_districts=num_district,
            voters_per_district=voters_per_district,
            alpha=alpha, beta=beta, seed=int(rng.integers(1e9))
        )
        X.append(votes_per_district.astype(np.float32))
        Y.append((alpha, beta))
        true_params.append((alpha.astype(np.float32), float(beta)))
        num_districts.append(num_district)
        voters_per_districts.append(voters_per_district)
    return X, Y, true_params, num_districts, voters_per_districts

In [6]:
import scipy
import scipy.io as sio
from google.colab import drive
import os

drive.mount('/content/drive', force_remount=True)
mat_file_path = '/content/drive/MyDrive/DPM_grand.mat'
data=sio.loadmat(mat_file_path)

Mounted at /content/drive


In [7]:
X = data['CC']
alpha = data['alpha']
beta = data['beta']
print(X.shape)
x= X[0][10]
print(alpha[500])
print(x.shape)

(1, 1000)
[0.45315585 0.26845634 0.27838782]
(100, 3)


In [10]:
X = data['CC']
alpha = data['alpha']
beta = data['beta']
#dists = data['S']
#NUM_DISTRICTS = dists
NUM_DISTRICTS = X[0][0].shape[0]
#NUM_EXAMPLES = X[0].shape[0]
NUM_PARTIES = 3
VOTERS_PER_DISTRICT = int(np.sum(X[0][0][0]))

train_range = range(0,500)
NUM_EXAMPLES = len(train_range)

X_np, Y, true_params = [], [], []
for i in train_range:
      #print(X_np[i], alpha[i], beta[0][i])
      X_np.append(X[0][i].astype(np.float32))
      Y.append((alpha[i], float(beta[0][i])))
      true_params.append((alpha[i].astype(np.float32), float(beta[0][i])))

In [11]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import RelaxedOneHotCategorical
import scipy.stats
import matplotlib.pyplot
import gc # Import garbage collector

from scipy.io import savemat
import numpy as np

EPOCHS = 25
BATCH_SIZE = 100  # Reduced batch size to mitigate OutOfMemoryError

#if __name__ == "__main__":
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("Device:", device)

# Clear GPU memory before starting training, if available
if device == 'cuda':
    print("Clearing GPU memory...")
    torch.cuda.empty_cache()
    gc.collect() # Force Python garbage collection
    print("GPU memory cleared.")
    print("GPU memory summary AFTER clearing cache:")
    print(torch.cuda.memory_summary())

print(f"VOTERS_PER_DISTRICT: {VOTERS_PER_DISTRICT}")
print("Training on GPU..." if device == 'cuda' else "Training on CPU...")
model, loss_history = train_encoder_decoder(
      X_np, true_params,
      num_examples=NUM_EXAMPLES,
      num_districts=NUM_DISTRICTS,
      num_parties=NUM_PARTIES,
      voters_per_district=VOTERS_PER_DISTRICT,
      device=device,
      epochs=EPOCHS,
      batch_size=BATCH_SIZE,
      lr=1e-3,
      gumbel_temp=0.25
    )

print("\nEncoder-decoder training complete.")

Device: cuda
Clearing GPU memory...
GPU memory cleared.
GPU memory summary AFTER clearing cache:
|===========================================================================|
|                  PyTorch CUDA memory summary, device ID 0                 |
|---------------------------------------------------------------------------|
|            CUDA OOMs: 0            |        cudaMalloc retries: 0         |
|===========================================================================|
|        Metric         | Cur Usage  | Peak Usage | Tot Alloc  | Tot Freed  |
|---------------------------------------------------------------------------|
| Allocated memory      |   7770 MiB |  12378 MiB |   1816 GiB |   1809 GiB |
|       from large pool |     16 MiB |     16 MiB |      0 GiB |      0 GiB |
|       from small pool |   7754 MiB |  12362 MiB |   1816 GiB |   1809 GiB |
|---------------------------------------------------------------------------|
| Active memory         |   7770 MiB |  12378

In [12]:

import scipy
import scipy.io as sio
from google.colab import drive
import os

drive.mount('/content/drive', force_remount=True)
mat_file_path = '/content/drive/MyDrive/DPM_grand.mat'
data2=sio.loadmat(mat_file_path)

#data2 = data

Xtest = data2['CC']
alpha_true_np = data2['alpha']
beta_true_np = data2['beta'][0]

#d = dists.min()
#x = X[0][i].astype(np.float32)
#n = x.shape[0]
#xd = x[n-d:n]
#print(xd)

Mounted at /content/drive


In [13]:
print(Xtest.shape)
print(alpha_true_np.shape)
print(beta_true_np.shape)

(1, 1000)
(1000, 3)
(1000,)


In [15]:
#xtest = Xtest[0][0]
#test_districts = xtest.shape[0]

#d = dists.min()
d=NUM_DISTRICTS
print(d)

test_range = range(0,1000)

X_test, Y_test, test_params = [], [], []
for i in test_range:
      #print(X_np[i], alpha[i], beta[0][i])
      x = Xtest[0][i].astype(np.float32)
      #n = x.shape[0]
      #xd = x[n-d:n]
      #distlist = np.random.choice(n, d, replace=False)
      #xd = x[distlist]
      #X_test.append(xd)
      X_test.append(x)
print(x.shape)

#alpha_test = alpha_true_np[test_range]
#beta_test = beta_true_np[test_range]

#Xout = data2['CC_test']
#x1 = Xout
#x1 = Xout[0][0]
#x2 = Xout[0][1]
#X_test.append(x1)
#X_test.append(x2)

X_test = np.stack(X_test)
X_test = np.ascontiguousarray(X_test)

print(X_test.shape)

with torch.no_grad():
      X_tensor = torch.tensor(X_test, dtype=torch.float32, device=device)
      alpha_pred_all, beta_pred_all, _ = model(X_tensor, VOTERS_PER_DISTRICT)
      alpha_pred = alpha_pred_all.cpu().numpy()
      beta_pred = beta_pred_all.cpu().numpy()

100
(100, 3)
(1000, 100, 3)


In [16]:
print(alpha_pred[500])
print(alpha_pred[499])
print(beta_pred[500])
#print(beta_pred[701])

[0.43289566 0.2871768  0.27992755]
[0.39745793 0.40636203 0.19618   ]
0.8551996


In [ ]:
beta_corr = scipy.stats.pearsonr(beta_test,beta_pred)
alpha_corr = scipy.stats.pearsonr(alpha_true_np, alpha_pred)
print (beta_corr)

a1 = alpha_test[:, 0]
a2 = alpha_test[:, 1]
a3 = alpha_test[:, 2]

b1 = alpha_pred[:, 0]
b2 = alpha_pred[:, 1]
b3 = alpha_pred[:, 2]

c1 = scipy.stats.pearsonr(a1, b1)
c2 = scipy.stats.pearsonr(a2, b2)
c3 = scipy.stats.pearsonr(a3, b3)

print(beta_corr)

print(c1)
print(c2)
print(c3)

PearsonRResult(statistic=np.float64(0.9168380711839677), pvalue=np.float64(1.561096104473217e-280))
PearsonRResult(statistic=np.float64(0.9168380711839677), pvalue=np.float64(1.561096104473217e-280))
PearsonRResult(statistic=np.float64(0.9682024317386893), pvalue=np.float64(0.0))
PearsonRResult(statistic=np.float64(0.965055816609527), pvalue=np.float64(0.0))
PearsonRResult(statistic=np.float64(0.9655750706054838), pvalue=np.float64(0.0))


In [ ]:
Xtest = data2['CC_test']

X_out = []
x1 = Xtest[0][0]
x2 = Xtest[0][1]
X_out.append(x1)
X_out.append(x2)
X_out = np.stack(X_out)
X_out = np.ascontiguousarray(X_out)

X_tensor = torch.tensor(X_out, dtype=torch.float32, device=device)
alpha_out, beta_out, _ = model(X_tensor, VOTERS_PER_DISTRICT)

print(alpha_out)
print(beta_out)

tensor([[0.2089, 0.5598, 0.2313],
        [0.4337, 0.1407, 0.4256]], device='cuda:0', grad_fn=<SoftmaxBackward0>)
tensor([0.7382, 0.7289], device='cuda:0', grad_fn=<AddBackward0>)


In [21]:
from scipy.io import savemat
import numpy as np
gradDPM_test = {"alpha_true": alpha_true_np, "alpha_pred1": alpha_pred, "beta_true": beta_true_np, "beta_pred1": beta_pred}
savemat("gradDPM_grand.mat", gradDPM_test)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
mat_file_path = '/content/drive/MyDrive/indianelections.mat'
data1=sio.loadmat(mat_file_path)

x1 = data1['X1']
x2 = data1['X2']

X1, X2 = [], []

for i in range(0,6):
  X1.append(x1[0][i].astype(np.float32))
  X2.append(x2[0][i].astype(np.float32))

In [ ]:
import random

device = 'cuda' if torch.cuda.is_available() else 'cpu'

d = dists.min()

s = X2[0].shape[0]
fulldist = range(0, s)

XX = []
for j in range(0, 100):
    distlist = random.sample(fulldist, d)
    x2 = np.round(X2[0][distlist])
    XX.append(x2)

XX = np.stack(XX)

with torch.no_grad():
      X_tensor = torch.tensor(XX, dtype=torch.float32, device=device)
      alpha_pred_all, beta_pred_all, _ = model(X_tensor, VOTERS_PER_DISTRICT)
      alpha_pred_np = alpha_pred_all.cpu().numpy()
      beta_pred_np = beta_pred_all.cpu().numpy()

In [ ]:
from scipy.io import savemat
import numpy as np
a = np.arange(20)
gradDPM_test = {"XX":XX, "alpha_pred": alpha_pred_np, "beta_pred": beta_pred_np}
savemat("indianelection_test.mat", gradDPM_test)

In [ ]:
print(alpha_pred_np)

[[0.10348187 0.8096274  0.08689076]
 [0.15909131 0.69526196 0.14564677]
 [0.47728977 0.14623222 0.37647808]
 [0.6258213  0.2021028  0.17207593]
 [0.5631691  0.26836914 0.1684617 ]
 [0.64065284 0.18044572 0.17890143]
 [0.1325792  0.5999755  0.26744524]
 [0.18783417 0.61939114 0.19277467]
 [0.24737684 0.16640766 0.5862155 ]
 [0.69350827 0.18019369 0.12629808]
 [0.10833643 0.30983645 0.58182716]
 [0.19102947 0.6338428  0.17512771]
 [0.1477511  0.77006626 0.08218271]
 [0.40896815 0.18982686 0.401205  ]
 [0.28941417 0.29538885 0.415197  ]
 [0.70299375 0.17296135 0.12404482]
 [0.20037672 0.43628398 0.36333936]
 [0.3882421  0.3912417  0.22051619]
 [0.23375483 0.18744923 0.5787959 ]
 [0.11084737 0.6554457  0.23370695]
 [0.7598189  0.12594093 0.11424018]
 [0.22772332 0.5450447  0.22723192]
 [0.6299707  0.17847358 0.19155574]
 [0.18531504 0.6732288  0.14145616]
 [0.12819123 0.7885775  0.08323126]
 [0.16581257 0.2682795  0.56590796]
 [0.11558993 0.22610904 0.65830106]
 [0.15221913 0.68796897 0.15